In [ ]:
%matplotlib ipympl

import sys
import os
sys.path.append("../src/")
import matplotlib.pyplot as plt

import importlib
import cdsaxs
import cdsaxs.loaders
import cdsaxs.loaders.load_data as loaders
import cdsaxs.plotting
import numpy as np

# if you are not actively developing the code, you can comment the rest of these out
importlib.reload(cdsaxs)
importlib.reload(cdsaxs.tools)
importlib.reload(cdsaxs.calculators)
importlib.reload(cdsaxs.data.data1d)
importlib.reload(cdsaxs.data.data2d)
importlib.reload(cdsaxs.data.dataset)
importlib.reload(cdsaxs.loaders)
importlib.reload(cdsaxs.data.metadata)
importlib.reload(cdsaxs.plotting)
importlib.reload(cdsaxs.plotting.plotting)
importlib.reload(cdsaxs.plotting._plotting_tools)
# importlib.reload(cdsaxs.plotting._plotly_tools)
importlib.reload(cdsaxs.reduction)
importlib.reload(cdsaxs.data.data2d)

from cdsaxs.plotting import _plotly_tools as plotly_tools
from cdsaxs.plotting import _plotting_tools as plotting_tools
from cdsaxs.plotting import plotting as cdsaxs_plotting

# cvs_path = os.path.abspath("data_W204_F2/W204_metadata.csv")
# dataset = loaders.LoadDataset_MetadataCSV("test", metadata_csv_filepath=cvs_path, data_name_pattern="{sample_phi_deg}", detector_type='pilatus')
# dataset.update_all_metadata({'center_px': (740, 490), 'sample_phi_offset_deg': -2})

# data = dataset.datas["0.0"]

## Loading Data

In [ ]:
from cdsaxs.loaders import load_data
data_dir = "data_W204_F2/"
filenames = load_data.filter_filenames(data_dir, filter_substrings=[['W204','0deg']], file_extension='tif')
filenames

In [ ]:
data = load_data.LoadData("data_W204_F2/W204_F2measure1_5.2m_16.1keV_num60_00deg_bpm0.417_id857181_combined.tif", # path to single file to load as Data2D
                          metadata={'sdd_cm': 500, 'energy_ev': 16100, 'exposure_time_s': 2, 'sample_phi_deg': 0}, # optional, assign metadata
                          user_params={'sample_name': 'W204 F2'}, # optional, assign user parameters
                          name='test data2d', # optional, name for the Data2D instance
                          filetype='tif', # optional, set the filetype to help the loader pick the right function to read the file, otherwise it will try to figure it out based on the file extension
                          detector_type='pilatus') # optional, it will try to extract count time and pixel size from the pilatus header, assuming 172 um pixel size if that fails

In [ ]:
type(data)

In [ ]:
dataset = load_data.LoadDataset(
    'test_dataset', # the dataset name is required
    data_dir, # directory where the data is located
    # filenames=filenames, # optional, list of filenames to load if you want to filter the data directory
    metadata_pattern="W204_F2measure1_{sdd_cm}m_{energy_ev}keV_num{num}_{sample_phi_deg}deg_bpm{bpm}_id{id}_combined.tif", # optional, filename pattern to extract metadata from the filenames
    metadata_scales={'sdd_cm': 100, 'energy_ev': 1000}, # optional, scales these metadata if they were in the filename with incorrect unnits
    data_name_pattern='W204 F2 at {sample_phi_deg} deg', # optional, pattern to follow for the data keys (each detector image in the dataset, will need to be unique), default is the filename
    metadata={'exposure_time_s': 2}, # optional, any extra metadata that applies to all data
    user_params={'sample_name': 'W204 F2'}, # optional, any unique user parameters that applies to all data
    verbose=True, # optional, will provide a status bar while files are loaded
    filetype='tif', # optional, specify the filetype, this will help filter the files if they have not been filtered already, if not provided the laoder will try to guess based on the file extension
    detector_type='pilatus'# optional, will try to extract count time and pixel size from the pilatus header, if this isn't possible it will assume 172 um as the pixel size
)

In [ ]:
type(dataset)

In [ ]:
list(dataset.datas.keys())

In [ ]:
dataset_csv = load_data.LoadDataset_MetadataCSV(
    "test dataset csv",
    metadata_csv_filepath="data_W204_F2/W204_metadata.csv",
    verbose=True,
    filetype='tif',
    detector_type='pilatus',
    data_name_pattern='W204 F2 at {sample_phi_deg} deg'
)

In [ ]:
type(dataset_csv)

In [ ]:
list(dataset_csv.datas.keys())

## Data2D

In [ ]:
data.metadata

In [ ]:
data.update_metadata(metadata={'exposure_time_s': 3}, overwrite=True)
data.metadata

In [ ]:
data.user_params

In [ ]:
data.update_user_params(params={'sample_name': 'W204 F2 0 degrees'}, overwrite=True)
data.user_params

In [ ]:
# note that the mask should be default mask out any points that are inf, -inf, or nan when the data is loaded unless this is overwritten by the user
# any of the standard colors accepted by matplotlib can be used to designate the color for any masked or inf/nan points
# 'transparent' can also be used as an accepted keyword for any of the masked regions

fig = data.plot_data(
    log_scale=True,
    cmap='viridis', # optional, switch to any accepted colormap in matplotlib
    aspect='equal', # optional, 'equal' will keep square pixels while 'auto' will reshape the detector image to fit the figure size
    vmin=None, # optional, set the colorbar range min, noting that vmin of 0 will not be valid for a log-scale plot
    vmax=None, # optional, set the colorbar range max, 
    color_mask='transparent', # optional, default is transparent, any points that are part of the existing mask
    color_inf='black', # optional, default is black, any -inf/inf points not in the mask and also any intensities less than or equal to 0 are shown as this color
    color_nan='red' # optional, default is red, any nan points not in the mask and also any intensities less than or equal to 0 are show as this color
)

In [ ]:
data.scale_data(1000) # normalize, add_to and subtract_from are available options
fig = data.plot_data()

In [ ]:
data.scale_by_metadata('exposure_time_s') # you can also normalize by any metadata or user_params
fig = data.plot_data()

In [ ]:
data.reset_intensity() # resets any normalization, scaling, adding, or subtracting that has been applied
fig = data.plot_data()

In [ ]:
data.rotate_image_ccw(1) # rotate the image by 90 degree steps counter clockwise (this is actually clockwise about the positive z axis)
fig = data.plot_data()

In [ ]:
# good to close plots occasionally for memory purposes
plt.close('all')

In [ ]:
data.flip_horizontally()
fig = data.plot_data()

In [ ]:
data.reset_image() # resets any rotations AND normalize/scale/add/subtract! will also reset the beam center to (0, 0) so be careful when working this into your workflow
fig = data.plot_data()

In [ ]:
data.update_metadata(dict(center_px=(740, 490)))
fig = data.plot_data()

In [ ]:
data.rotate_image(20, resampling_mode='bilinear') # the image can also be rotated about the beam center (if set) by a specified number of degrees
fig = data.plot_data()

In [ ]:
data.reset_image()
data.update_metadata(dict(center_px=(740, 490))) # make sure to reset the beam center after a reset image!
fig = data.plot_data()

In [ ]:
box_limits_qdy, box_limits_qdx = data.get_box_dims_size( # get the index limits for an roi centered (or offset) on the beam center with the specified pixel size
    size_qdy_px = 20, # width of box vertically
    size_qdx_px = 500, # width of box horizontally
    shift_box_qdy_px=0, # shift the box vertically by this number of pixels
    shift_box_qdx_px=0, # shift the box horizontally by this number of pixels
)
box_limits_qdy, box_limits_qdx

In [ ]:
box_limits_qdy, box_limits_qdx = data.get_box_dims_qrange(
    range_qdy=(-0.01, 0.01),
    range_qdx=(0.1, -0.1))
box_limits_qdy, box_limits_qdx

In [ ]:
qslice, (fig, fig_box, fig_slice, fig_backgrounds) = data.integrate_box(
    limits_qdy_px=20, # you can also give this an index range like (730, 750) otherwise the box will be centered or offset from beam center
    limits_qdx_px=500, # you can also give this an index range like (240, 740) otherwise the box will be centered or offset from beam center
    mode='sum',
    axis=None, # other accepted inputs are 'qdy', 1, 'qdx' or 0; this is the axis that will be integrated across; if you leave it to None the function will assume to integrate across the shorter side of the integration box
    shift_box_qdy_px=0,
    shift_box_qdx_px=0,
    show_plot=True,
    subtract_background_offset=None, # change this to a single integer or list of integers to grab background ROIs for a subtraction
    plotting_kwargs={}, # any keyword arguments to matplotlib.pyplot.errorbar or to cdsaxs.plotting.plot_data2d_integrate_box can be passed using this dictionary
)


In [ ]:
plt.close('all')

In [ ]:
qslice, (fig, fig_box, fig_slice, fig_backgrounds) = data.integrate_box(
    limits_qdy_px=20, # you can also give this an index range like (730, 750) otherwise the box will be centered or offset from beam center
    limits_qdx_px=500, # you can also give this an index range like (240, 740) otherwise the box will be centered or offset from beam center
    mode='sum',
    axis=None, # other accepted inputs are 'qdy', 1, 'qdx' or 0; this is the axis that will be integrated across; if you leave it to None the function will assume to integrate across the shorter side of the integration box
    shift_box_qdy_px=0,
    shift_box_qdx_px=0,
    show_plot=True,
    subtract_background_offset=[-150, -90], # change this to a single integer or list of integers to grab background ROIs for a subtraction (the offset direction follows qdy or qdx axis)
    plotting_kwargs={
        'show_backgrounds': True,
        'show_legend': True,
        'color_avg_background': 'grey',
        'color_slice': 'black',
        'ms': 2,
    }, # any keyword arguments to matplotlib.pyplot.errorbar or to cdsaxs.plotting.plot_data2d_integrate_box can be passed using this dictionary
)

In [ ]:
plt.close('all')
peaks, peaks_q, fig = data.find_peaks2D( # you can find 2D sets of peaks with this function
    limits_qdy_px=100, # or you can set an index range!
    limits_qdx_px=500,
    log_scale=True,
    refinement_size=7, # box around each peak sent to the Gaussian refinement function
    show_plot=True, 
    zoom_plot=True, # if set to True, the search box with a small border will be shown in the plot to more easily see the peaks found, otherwise if set to False they will all be overlaid on the original detector image
    plotting_kwargs={},
    min_distance=10, # see documentation for scikit-image peak_local_max() for all keyword arguments you can pass through
    threshold_abs=2, # if log_scale is set to True, the data is sent to the peak finding algorithm after taking log10 so make sure these values make sense for data after the transformation
)


In [ ]:
plt.close('all')
peaks, peaks_q, fig = data.find_peaks2D_one_axis( # you can find peaks in a 2D region of interest but this function limits only one peak in each row/column along the peak axis
    limits_qdy_px=100, # or you can set an index range!
    limits_qdx_px=500,
    peak_axis='qdx',
    integration_mode='sum',
    algorithm='scikit', # you can also switch to the legacy peak finder using 'scipy'
    log_scale=True,
    refinement_size=7, # box around each peak sent to the Gaussian refinement function
    show_plot=True, 
    zoom_plot=True, # if set to True, the search box with a small border will be shown in the plot to more easily see the peaks found, otherwise if set to False they will all be overlaid on the original detector image
    plotting_kwargs={},
    min_distance=10, # see documentation for scikit-image peak_local_max() for all keyword arguments you can pass through
    threshold_abs=2, # if log_scale is set to True, the data is sent to the peak finding algorithm after taking log10 so make sure these values make sense for data after the transformation
)

In [ ]:
new_center, fig = data.find_beam_center_from_peaks( # make sure the peaks you find are symmetric around the beam stop 
    size_qdy_px=20,
    size_qdx_px=300,
    update=True, # whether to update the metadata for beam center, default is True
    beam_center_guess=None, # it will use the current beam center as the starting guess, which does need to be somewhat close as the function assumes peak orders on either side based on this guess; you can also give a new guess here on the fly
    show_plot=True,
    zoom_plot=True,
    ignore_peaks=[5, 6], # ignore some of the found peaks when determining beam center, in this case we will ignore the partially cut first order peaks (I am giving the keyword arguments the index that the peaks are in the list of all peaks, this might take some guessing at first)
    min_distance=10,
    plotting_kwargs={
        'color_integration_box': 'yellow',
        'color_peaks': 'red',
        'color_beam_center': 'red'}
)

In [ ]:
new_sdd_avg, new_sdd_std, fig = data.find_sdd_from_reference_peaks(
    pitch_nm = 100,
    size_qdy_px=20,
    size_qdx_px=300,
    update=True,
    show_plot=True,
    peak_orders=None,
    ignore_orders=[1, -1], # try commenting out this line to see the effects of a partially cutoff peak on the SDD!
    min_distance=10,
)

In [ ]:
data.reset_image()
data.update_metadata(dict(center_px=new_center))
data.rotate_image(3) # offset the image to make sure the function is working properly
angle, fig = data.find_detector_rotation_correction( # angle  reference frame is a counterclockwise rotation about the POSITIVE z-axis (into the screen) so that this angle can be fed directly to the rotate_image method to correct the image
    size_qdy_px=50,
    size_qdx_px=300,
    show_plot=True,
    zoom_plot=True,
    min_distance=10)
print(angle)

In [ ]:
data.rotate_image(angle)
fig = data.plot_data()

In [ ]:
data.reset_image()
data.update_metadata(dict(center_px=new_center))
fig=data.plot_data()

## Dataset

In [ ]:
dataset.datas # individual data instances are still stored in dataset.datas

In [ ]:
dataset.name

In [ ]:
dataset.__dict__ # dataset no longer includes integrated datasets or reduced datasets/slices!

In [ ]:
dataset.add_data(data) # you can add one or more Data2D instances to a dataset

In [ ]:
dataset.datas

In [ ]:
dataset.remove_data('test data2d') # you can remove it by either passing the data instance again or using the name

In [ ]:
dataset.datas

In [ ]:
one_data = dataset.datas['W204 F2 at -10.0 deg']
one_data.metadata

In [ ]:
dataset.update_all_metadata(
    metadata=dict(center_px=new_center),
    overwrite=True,
    keys=None, # you can pass a list of keys if you only want to update some of the data
)
one_data.metadata

In [ ]:
# new way to filter out keys by a single metadata parameter
keys = dataset.filter_data_by_metadata('sample_phi_deg', (-20, 20))
keys

In [ ]:
keys = dataset.filter_data_by_metadata('sample_phi_deg', [10, 0])
keys

In [ ]:
keys = dataset.filter_data_by_metadata('sample_phi_deg', 0)
keys

In [ ]:
# you can normalize, add, subtract, scale, rotate counterclockwise, and reset all images or just intensities across the whole dataset or a subset of the dataset by giving a list of filtered data keys
fig = one_data.plot_data()
dataset.scale_all_data(2)
fig = one_data.plot_data()

In [ ]:
dataset.reset_all_data_intensity()
fig = one_data.plot_data()

In [ ]:
integrated_dataset = dataset.integrate_dataset(
    limits_qdy_px=10,
    limits_qdx_px=500,
    mode='sum',
    axis=0)

## Integrated Dataset & QSlice
The integrated dataset class is really a container for many QSlice instances created by integrated a dataset.

In [ ]:
integrated_dataset.__dict__

In [ ]:
fig = integrated_dataset.plot_data(
    q_axis='qdx',
    y_axis='sample_phi_deg',
    log_scale=True,
    cmap='viridis',
    vmin=None,
    vmax=None,
    filter_by_q={},
    filter_by_metadata={'sample_phi_deg': (-40, 40)},
    s=2
)

In [ ]:
fig = integrated_dataset.plot_data(
    q_axis='qdx',
    y_axis='sample_phi_deg',
    log_scale=True,
    cmap='viridis',
    vmin=None,
    vmax=None,
    filter_by_q={},
    filter_by_metadata={'sample_phi_deg': 0},
    s=2
)

In [ ]:
fig = integrated_dataset.plot_data(
    q_axis='qdx',
    y_axis='sample_phi_deg',
    log_scale=True,
    cmap='viridis',
    vmin=None,
    vmax=None,
    filter_by_q={},
    filter_by_metadata={'sample_phi_deg': [0, 10, -10, 20, -20, 30, -30]},
    s=2
)

## Reduced Dataset and ReducedData1D

In [ ]:
reduced_dataset = cdsaxs.reduction.reduce_dataset(integrated_dataset)

In [ ]:
reduced_dataset.data

In [ ]:
reduced_data1d = reduced_dataset.data[0]

In [ ]:
fig = reduced_data1d.plot_data(
    q_axis=None,
    log_scale=True,
    show_legend=True,
    xlim=None,
    ylim=None,
    fmt='o-',
    label=reduced_data1d.sample_phi_deg # you can pass any keyword arguments for matplotlib.errorbar in as keyword arguments to this function
)

## Reduced Slices and ReducedData1DSlice

In [ ]:
q_values = np.arange(1, 5)*2*np.pi/1000
reduced_slices, fig = cdsaxs.reduction.slice_reduced_dataset(reduced_dataset, q_values=q_values, q_widths=0.002, plotting_kwargs=dict(interpolated_data=False, s=4, slice_lw=1))

In [ ]:
reduced_slices.data

In [ ]:
reduced_slice = reduced_slices.data[3]

In [ ]:
fig = reduced_slice.plot_data(fmt='o')

In [ ]:
reduced_slice.Iq

In [ ]:
reduced_slice.qsz

In [ ]:
reduced_dataset = cdsaxs.reduction.reduce_dataset(integrated_dataset)

In [ ]:
import cdsaxs.plotting.plotting
fig = cdsaxs.plotting.plotting.plot_reduced_dataset(reduced_dataset, interpolated_data=True)

In [ ]:
q_values = np.arange(1, 5)*2*np.pi/1000
reduced_slices, fig = cdsaxs.reduction.slice_reduced_dataset(reduced_dataset, q_values=q_values, q_widths=0.002, plotting_kwargs=dict(interpolated_data=False, s=4, slice_lw=1))

In [ ]:
importlib.reload(cdsaxs)
importlib.reload(cdsaxs.plotting.plotting)
importlib.reload(cdsaxs.plotting._plotting_tools)
importlib.reload(cdsaxs.data.dataset)
importlib.reload(cdsaxs.reduction)
importlib.reload(cdsaxs.data)
importlib.reload(cdsaxs.data.data1d)
importlib.reload(cdsaxs.data.reduced_data1d)

In [ ]:
cdsaxs.plotting.plotting.plot_reduced_slices(reduced_slices, offset_order=2)